# 02. YOLOv8 학습 실습 — Roboflow `stamp` 데이터셋

**목표** — 사전학습 `yolov8n.pt`를 제조 현장 데이터(신발 + 스탬프 2개 클래스)로 파인튜닝하고, 평가 지표를 해석한다.
더 나아가 **모델(YOLOv8n vs YOLO11n)** 과 **에폭 수**를 바꿔 성능 범위를 비교한다.

**데이터 준비** (직접 1회)
1. https://universe.roboflow.com/warisara-kaewsuwan-cf2hs/stamp-bcrhe 접속 → Download Dataset → 형식 **YOLOv8** → zip 다운로드
2. 압축을 풀어서 `CV-YOLO/data/stamp/` 아래에 `train/ valid/ test/ data.yaml`이 오도록 둔다.

In [ ]:
from pathlib import Path
import yaml, torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

ROOT = Path.cwd().parent
STAMP = ROOT / "data" / "stamp"
RUNS = ROOT / "runs" / "stamp"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
assert (STAMP / "data.yaml").exists(), f"{STAMP}/data.yaml 이 없습니다. 위 '데이터 준비'를 먼저 하세요."

## 1. 데이터 확인
Roboflow가 만든 `data.yaml`은 상대경로(`../train/images`)라서 실행 위치에 따라 깨진다.
→ 절대경로 버전(`data_local.yaml`)을 새로 만들어 쓴다.

In [ ]:
cfg = yaml.safe_load((STAMP / "data.yaml").read_text(encoding="utf-8"))
print("원본:", {k: cfg[k] for k in ("train", "val", "names") if k in cfg})

local = {"path": STAMP.as_posix(), "train": "train/images", "val": "valid/images",
         "test": "test/images", "names": cfg["names"]}
DATA = STAMP / "data_local.yaml"
DATA.write_text(yaml.safe_dump(local, allow_unicode=True), encoding="utf-8")

def count(split):
    imgs = list((STAMP / split / "images").glob("*"))
    labels = list((STAMP / split / "labels").glob("*.txt"))
    boxes = pd.Series([int(l.split()[0]) for f in labels for l in f.read_text().splitlines() if l.strip()])
    names = cfg["names"] if isinstance(cfg["names"], list) else list(cfg["names"].values())
    return {"split": split, "images": len(imgs),
            **{n: int((boxes == i).sum()) for i, n in enumerate(names)}}

stats = pd.DataFrame([count(s) for s in ("train", "valid", "test")]); stats

In [ ]:
# 라벨이 어떻게 생겼는지 샘플 4장 확인
import cv2
samples = sorted((STAMP / "train" / "images").glob("*"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, samples):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for line in (STAMP / "train" / "labels" / (p.stem + ".txt")).read_text().splitlines():
        c, cx, cy, bw, bh = map(float, line.split()[:5])
        color = (255, 0, 0) if int(c) == 0 else (0, 200, 0)
        cv2.rectangle(im, (int((cx-bw/2)*w), int((cy-bh/2)*h)), (int((cx+bw/2)*w), int((cy+bh/2)*h)), color, 2)
    ax.imshow(im); ax.axis("off")
plt.suptitle("train labels (red=class0, green=class1)"); plt.tight_layout(); plt.show()

**의견** — (실행 후 작성) 클래스별 박스 수 균형, 이미지 크기, 물체 크기를 확인한다.

## 2. 기본 학습 — YOLOv8n, 20 epochs (과제 설정 그대로)

In [ ]:
model = YOLO("yolov8n.pt")
model.train(data=str(DATA), epochs=20, imgsz=640, batch=16, device=DEVICE,
            project=str(RUNS), name="v8n_e20", exist_ok=True, workers=2, seed=42, plots=True)

In [ ]:
Image.open(RUNS / "v8n_e20" / "results.png")

## 3. 평가 지표 해석

In [ ]:
best = YOLO(str(RUNS / "v8n_e20" / "weights" / "best.pt"))
metrics = best.val(data=str(DATA), device=DEVICE, project=str(RUNS), name="v8n_e20_val", exist_ok=True)

names = best.names
rows = [{"Class": "all", "Precision": metrics.box.mp, "Recall": metrics.box.mr,
         "mAP@.5": metrics.box.map50, "mAP@.5:.95": metrics.box.map}]
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append({"Class": names[int(c)], "Precision": p, "Recall": r, "mAP@.5": ap50, "mAP@.5:.95": ap})
table = pd.DataFrame(rows).round(3)
print(table.to_markdown(index=False))
print("mAP75:", round(metrics.box.map75, 3))

| 지표 | 의미 |
|---|---|
| Precision | 모델이 찾았다고 한 것 중 맞은 비율 = TP / (TP + FP) |
| Recall | 실제 정답 중 찾아낸 비율 = TP / (TP + FN) |
| mAP@.5 | IoU 0.5 기준 AP의 클래스 평균 — "대충 위치를 맞췄나" |
| mAP@.5:.95 | IoU 0.5~0.95 평균 — "박스를 얼마나 딱 맞게 쳤나" (더 엄격) |

**의견** — (실행 후 작성) mAP@.5는 높은데 mAP@.5:.95가 낮은 클래스는 위치는 찾지만 박스 경계가 덜 정확하다는 뜻이다.

In [ ]:
Image.open(RUNS / "v8n_e20" / "confusion_matrix_normalized.png")

## 4. test 이미지로 예측 확인

In [ ]:
test_imgs = sorted((STAMP / "test" / "images").glob("*"))[:6]
preds = best.predict([str(p) for p in test_imgs], conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(axes.flat, preds):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(Path(r.path).name[:30], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
preds[0].boxes

## 5. 추가 실험 — 모델과 에폭 수에 따른 성능 범위

| 실험 | 모델 | epochs | 확인하려는 것 |
|---|---|---|---|
| A | YOLOv8n | 20 | 기준 (과제 설정) |
| B | YOLOv8n | 50 | 더 오래 학습하면 좋아지나? |
| C | YOLO11n | 20 | 최신 모델로 바꾸면? (다른 모델 적용) |
| D | YOLOv8s | 20 | 모델을 키우면? (속도와 맞바꿈) |

In [ ]:
EXPS = [("v8n_e20", "yolov8n.pt", 20), ("v8n_e50", "yolov8n.pt", 50),
        ("11n_e20", "yolo11n.pt", 20), ("v8s_e20", "yolov8s.pt", 20)]
summary = []
for name, weights, ep in EXPS:
    run = RUNS / name
    if not (run / "weights" / "best.pt").exists():
        YOLO(weights).train(data=str(DATA), epochs=ep, imgsz=640, batch=16, device=DEVICE,
                            project=str(RUNS), name=name, exist_ok=True, workers=2, seed=42, plots=True, verbose=False)
    m = YOLO(str(run / "weights" / "best.pt"))
    v = m.val(data=str(DATA), split="test", device=DEVICE, verbose=False, plots=False,
              project=str(RUNS), name=f"{name}_test", exist_ok=True)
    summary.append({"exp": name, "model": weights, "epochs": ep,
                    "params(M)": round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2),
                    "P": v.box.mp, "R": v.box.mr, "mAP50": v.box.map50, "mAP50-95": v.box.map,
                    "infer_ms": v.speed["inference"]})
exp_df = pd.DataFrame(summary).round(3)
exp_df.to_csv(ROOT / "results" / "stamp_experiments.csv", index=False)
exp_df

In [ ]:
ax = exp_df.plot.bar(x="exp", y=["mAP50", "mAP50-95"], rot=0, figsize=(8, 4), color=["#94a3b8", "#3b82f6"])
ax.set_ylim(0, 1.05); ax.set_title("stamp test set — model / epoch comparison")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "report" / "images" / "stamp_experiments.png", dpi=150); plt.show()

**의견** — (실행 후 작성) 어떤 조합이 정확도/속도 균형이 가장 좋은지, 에폭을 늘린 효과가 있었는지 정리한다.

## 정리
- 사전학습 모델 + 1천 장 정도의 데이터로도 20 에폭 안에 높은 mAP@.5에 도달한다 (전이학습의 효과).
- 다음 단계: 직접 찍은 사진으로 **정상/불량 영역**을 학습 → `03_my_defect_detection.ipynb`